In [ ]:
from massspecgym.data.data_module import MassSpecDataModule
from massspecgym.data.datasets import RetrievalDataset
from massspecgym.data.transforms import InMemCachedMolTransform
from chemembed_transforms import (
    ChemEmbedSpecTransform,
    molformer_factory
)
from ann_model_massspecgym import AnnRetrieval
from pytorch_lightning import Trainer
import pandas as pd

In [ ]:
# Prepare test data

# MoLFormer (768 dims)
mol_transform_factory = molformer_factory
embedding_name = "molformer"
mol_embedding_dim = 768

selected_molecular_embedding = InMemCachedMolTransform(
    # cache_pth=Path(f"data/{embedding_name}_cache.pkl"),
    mol_transform_factory=mol_transform_factory,
    verbose=True
)

In [ ]:
# Load model
model_name = "ANN_trained_molformer.ckpt"
model = AnnRetrieval.load_from_checkpoint(f"models/{model_name}")
model.eval()

test_dataset = RetrievalDataset(
    candidates_pth="test.json",
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
)

data_module_test = MassSpecDataModule(
    dataset=test_dataset,
    batch_size=32,
)

data_module_test.setup(stage="test")

# Init trainer
trainer = Trainer(
    accelerator="auto",
    max_epochs=15,
    log_every_n_steps=1
)

results = trainer.test(model, datamodule=data_module_test)

print(results)

pd.DataFrame(results).to_csv("results.csv", index=False)